# NCM Basics

It is recommended to read follow this notebook to grasp the core workflow and fundamentals of training and evaluating Neural Causal Models (NCMs):

The following steps are shown:
1. Define a Structural Causal Model (SCM)
2. Generate synthetic data
3. Train min/max NCM models
4. Evaluate causal effects

First we start with basic binary envorionments and the we advance to continuous environments.

In [ ]:
import numpy as np
import pandas as pd
import torch as T
import matplotlib.pyplot as plt
import sys
import os

sys.path.insert(0, os.path.abspath("binary"))

from binary.models.ncm import NCM
from binary.training.train_minmax import train_minmax_models
from binary.inference.do_intervention import do_intervention

# Binary SCM Example

We define a simple **backdoor causal graph**:

Z → X → Y  
Z → Y

- Z: Confounder
- X: Treatment
- Y: Outcome

All variables are binary.

Feel free to adjust it to your preference.


In [ ]:

def generate_binary_scm(n_samples=3000, seed=0):
    rng = np.random.RandomState(seed)
    
    Z = rng.binomial(1, 0.5, size=n_samples)
    
    X = np.array([rng.rand() < (0.7 if z == 1 else 0.3) for z in Z]).astype(int)

    Y = np.array([rng.rand() < (0.9 if (x == 1 and z == 1)
                                   else 0.6 if (x == 1 and z == 0)
                                   else 0.3 if (x == 0 and z == 1) 
                                   else 0.1
                                   )
                    for x,z in zip(X,Z)], dtype=int)

    return pd.DataFrame(dict(Z=Z, X=X, Y=Y))

data = generate_binary_scm()
data.head()

## Training NCM Models

We train two models:

- **Min Model** → lower bound
- **Max Model** → upper bound

But only one function needs to be executed.

In [ ]:
min_model = NCM("binary/graphs/backdoor.cg")
max_model = NCM("binary/graphs/backdoor.cg")

opt_min = T.optim.Adam(min_model.parameters(), lr=4e-3)
opt_max = T.optim.Adam(max_model.parameters(), lr=4e-3)

gap_hist = train_minmax_models(
    epoch_num=500,
    m=200,
    data=data,
    min_model=min_model,
    max_model=max_model,
    optim_min=opt_min,
    optim_max=opt_max
)


In [ ]:
plt.plot(gap_hist)
plt.title("Gap over training")
plt.xlabel("Epoch")
plt.ylabel("|ATE_max - ATE_min|")
plt.grid()
plt.show()

## Binary Intervention Comparison

We compare the interventional probability \(P(Y=1 \mid do(X=x))\) from the SCM with estimates from trained NCMs.

For \(x \in \{0,1\}\), we:
- compute the ground truth using the SCM,
- estimate the same quantity using the Min and Max NCM models.

The bar plot shows:
- **SCM** (blue): true causal effect,
- **Min model** (red): lower bound estimate,
- **Max model** (orange): upper bound estimate.

The gap between Min and Max reflects uncertainty, while alignment with the SCM indicates accurate estimation.

In [ ]:
BLUE = "#1f77b4"
RED = "#d62728"
ORANGE = "#ff7f0e"


min_model.eval()
max_model.eval()

# =========================
# SCM (Do X=x)
# If you adjusted the original SCM, you have to adjust it here as well but keep the X = np.full part to perform intervention
# =========================
def scm_do_x(n, x, seed):
    rng = np.random.RandomState(seed)
    
    Z = rng.binomial(1, 0.5, size=n)
    
    X = np.full(n, x)

    Y = np.array([rng.rand() < (0.9 if (x == 1 and z == 1)
                                   else 0.6 if (x == 1 and z == 0)
                                   else 0.3 if (x == 0 and z == 1) 
                                   else 0.1
                                   )
                    for x,z in zip(X,Z)], dtype=int)

    return pd.DataFrame(dict(Z=Z, X=X, Y=Y))

gt = []
min_vals = []
max_vals = []

for x in [0,1]:

    # SCM
    gt.append(scm_do_x(20000, x, seed=0)["Y"].mean())

    # NCM
    # Here we simulate do intervention on the trained NCM
    min_vals.append(
        do_intervention(min_model, ['X'], x, 'Y', m=10000)
    )

    max_vals.append(
        do_intervention(max_model, ['X'], x, 'Y', m=10000)
    )

x_pos = np.array([0,1])
width = 0.25

plt.figure(figsize=(6,5))

# SCM
plt.bar(x_pos - width, gt, width, color=BLUE, label="SCM")

# MIN MODEL
plt.bar(x_pos, min_vals, width, color=RED, label="Min Model")

# MAX MODEL
plt.bar(x_pos + width, max_vals, width, color=ORANGE, label="Max Model")

# Labels
plt.xticks(x_pos, ["do(X=0)", "do(X=1)"])
plt.ylabel("P(Y=1 | do(X))")

plt.title("Backdoor: SCM vs Min/Max Model (seed=0)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

## Final Result

If everything finished correctly, you should get a figure containing multiple colored bars.

This figure compares the true interventional probability \(P(Y=1 \mid do(X=x))\) from the SCM with the estimates from the Min and Max NCM models.

- The **blue bars** represent the ground truth (SCM).
- The **red (Min)** and **orange (Max)** bars define the learned lower and upper bounds.

The results show that both models closely match the SCM for \(do(X=0)\) and \(do(X=1)\).  
Additionally, the gap between Min and Max is very small, indicating low uncertainty in the estimated causal effect.

Overall, this demonstrates that the NCM successfully learns the underlying causal relationship and accurately approximates the true intervention.

# Continuous SCM Example

We now use continuous variables:

Z → X → Y  
Z → Y

All variables are real-valued.

In [ ]:
import sys

for k in list(sys.modules.keys()):
    if k.startswith("inference") or k.startswith("models") or k.startswith("training"):
            del sys.modules[k]

sys.path.insert(0, os.path.abspath("continuous"))

from models.ncm import NCM
from training.train_minmax import train_minmax_models
from inference.do_intervention import ncm_do

## Continuous SCM Data Generation

In this step, we generate synthetic data from a continuous structural causal model (SCM).

The variables are defined as follows:
- \(Z\) is drawn uniformly from the interval \([0, 50]\)
- \(X\) depends linearly on \(Z\) with added Gaussian noise
- \(Y\) depends non-linearly on \(Z\), linearly on \(X\), and includes additional noise

This creates a **non-linear causal relationship** with a confounding structure via \(Z\).

The generated dataset is then used for training and evaluating the neural causal models (NCMs).

In [ ]:
def generate_continuous_scm(n=100000, seed=0):

    rng = np.random.RandomState(seed)

    eps_X = rng.normal(0, 1, size=n)
    eps_Y = rng.normal(0, 1, size=n)

    Z = rng.uniform(0, 50, size=n)
    X = 0.5 * Z + 10 * eps_X
    Y = 0.01 * (50 - Z * Z) + 0.5 * X + 30 * eps_Y

    return pd.DataFrame({'X': X, 'Z': Z, 'Y': Y})

data = generate_continuous_scm()
data.head()

## Normalization

Continuous models require normalization for stable training.

In [ ]:
mean = data.mean()
std = data.std()

data_norm = (data - mean) / std

## Training NCM Models

The training does not change for continuous cases, we still train a min and max model

In [ ]:
min_model = NCM("continuous/graphs/backdoor.cg", hidden_dim=64)
max_model = NCM("continuous/graphs/backdoor.cg", hidden_dim=64)

opt_min = T.optim.Adam(min_model.parameters(), lr=5e-4)
opt_max = T.optim.Adam(max_model.parameters(), lr=5e-4)

gap_hist = train_minmax_models(
    min_model=min_model,
    max_model=max_model,
    optim_min=opt_min,
    optim_max=opt_max,
    data=data_norm,
    epoch_num=500,
    batch_size=1024,
    m=300,
    treatment_var='X',
    outcome_var='Y',
    x_values=[-1, 0, 1]
)

## Gap Graph

As mentioned in the thesis, the gap graph shows us the uncertainty between min and max models. However, do not take the solution as a granted result. It is necesarry to compare the actual causal effect with the ground-truth.

In [ ]:
plt.plot(gap_hist)
plt.title("Continuous Gap")
plt.grid()
plt.show()

## Continuous Intervention Bounds (Min/Max Models)

We compare the true interventional expectation \(E[Y \mid do(X=x)]\) from the SCM with the bounds estimated by the Min and Max NCM models.

- The **black curve** represents the ground truth from the SCM.
- The **blue curve** (Min Model) and **red curve** (Max Model) define lower and upper bounds.

A narrow gap indicates confident estimation, while a wider gap reflects ambiguity in the causal effect.

In [ ]:
BLUE = "#1f77b4"
RED = "#d62728"
ORANGE = "#ff7f0e"


min_model.eval()
max_model.eval()

def scm_continuous_do_X(n, x, seed=0):

    rng = np.random.RandomState(seed)

    eps_Y = rng.normal(0, 1, size=n)
    Z = rng.uniform(0, 50, size=n)

    X = np.full(n, x)
    Y = 0.01 * (50 - Z * Z) + 0.5 * X + 30 * eps_Y

    return Y.mean()


raw_data = generate_continuous_scm(100000)
mean = raw_data.mean()
std = raw_data.std()

x_vals = np.linspace(-2, 2, 50)

scm_vals = []
min_vals = []
max_vals = []

for x in x_vals:
    x_raw = x * std["X"] + mean["X"]

    # SCM (raw → norm)
    scm_raw = scm_continuous_do_X(100000, x_raw)
    scm_norm = (scm_raw - mean["Y"]) / std["Y"]

    scm_vals.append(scm_norm)

    #we use ncm_do to simulate intervention within our NCM
    min_vals.append(ncm_do(min_model, x))
    max_vals.append(ncm_do(max_model, x))

scm_vals = np.array(scm_vals)
min_vals = np.array(min_vals)
max_vals = np.array(max_vals)

# =========================
# PLOT
# =========================
plt.figure(figsize=(6,4))

# SCM
plt.plot(x_vals, scm_vals, color="black", linewidth=3, label="SCM")

# Min / Max
plt.plot(x_vals, min_vals, "--", color=BLUE, label="Min Model")
plt.plot(x_vals, max_vals, "-", color=RED, label="Max Model")

plt.xlabel("X")
plt.ylabel("E[Y | do(X=x)]")

plt.title("Continuous Backdoor: Min/Max Bounds")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Final Result

This plot shows the expected outcome \(E[Y \mid do(X=x)]\) for the true SCM and the learned Min and Max models across a range of normalized intervention values.

The **black curve** represents the ground truth causal effect, while the **blue dashed** and **red** curves correspond to the learned lower and upper bounds.

The results should demonstrate that both models closely follow the true SCM over the entire range. The gap between the Min and Max models should be small, indicating low uncertainty and a well-identified causal effect.

Overall, the NCM should successfully captures the underlying continuous causal relationship and provide tight bounds around the true intervention.